In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, inspect, text
import urllib
from datetime import datetime

# --- CONFIGURATION ---
project_dir = "C:\\Users\\WORK-PC\\Documents\\Python Training\\"
csv_filename = "NetflixBusiness.csv"

# Connection string from the C# snippet adapted for Python (SQLAlchemy)
server = 'DESKTOP-GO1QVHO'
database = 'EmadeDev'
connection_string = f"DRIVER=ODBC Driver 17 for SQL Server;SERVER={server};DATABASE={database};Trusted_Connection=yes"

# Create SQLAlchemy engine
params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

def migrate_netflix_business():
    # Define the full path to the CSV file
    file_path = os.path.join(project_dir, csv_filename)
    
    if not os.path.exists(file_path):
        print(f"Error: CSV file not found at {file_path}")
        return

    print(f"Migrating {csv_filename} to database...")
    
    try:
        # 1. Define table name: Remove .csv and append REV
        table_name = os.path.splitext(csv_filename)[0] + "REV"
        
        print(f"--- Migrating {csv_filename} to table {table_name} ---")
        
        # 2. Read CSV into DataFrame from the project_dir
        df = pd.read_csv(file_path, encoding='latin1')
        
        # 3. Check if table exists and handle schema
        inspector = inspect(engine)
        table_exists = inspector.has_table(table_name)
        
        if table_exists:
            # Table exists - append the new data
            print(f"Table {table_name} exists. Appending data...")
            df.to_sql(table_name, con=engine, if_exists='append', index=False)
        else:
            # Table doesn't exist - create it with the data
            print(f"Table {table_name} does not exist. Creating new table...")
            df.to_sql(table_name, con=engine, if_exists='replace', index=False)
        
        print(f"Successfully migrated {len(df)} rows to {table_name}.")
        
    except Exception as e:
        print(f"Error migrating {csv_filename}: {e}")

if __name__ == "__main__":
    migrate_netflix_business()

Migrating NetflixBusiness.csv to database...
--- Migrating NetflixBusiness.csv to table NetflixBusinessREV ---
Table NetflixBusinessREV does not exist. Creating new table...
Successfully migrated 8807 rows to NetflixBusinessREV.


In [5]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 1. Count the number of Movies vs TV Shows
    q1 = """
    SELECT type, COUNT(*) as count
    FROM NetflixBusinessREV
    GROUP BY type
    """
    execute_query(q1, "1. Count the number of Movies vs TV Shows")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 1. Count the number of Movies vs TV Shows ---
   type  count
TV Show   2676
  Movie   6131


In [6]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 2. Find the most common rating for movies and TV shows
    q2 = """
    SELECT type, rating, COUNT(*) as count
    FROM NetflixBusinessREV
    WHERE rating IS NOT NULL
    GROUP BY type, rating
    ORDER BY type, count DESC
    """
    execute_query(q2, "2. Find the most common rating for movies and TV shows")


if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 2. Find the most common rating for movies and TV shows ---
   type   rating  count
  Movie    TV-MA   2062
  Movie    TV-14   1427
  Movie        R    797
  Movie    TV-PG    540
  Movie    PG-13    490
  Movie       PG    287
  Movie    TV-Y7    139
  Movie     TV-Y    131
  Movie     TV-G    126
  Movie       NR     75
  Movie        G     41
  Movie TV-Y7-FV      5
  Movie       UR      3
  Movie    NC-17      3
  Movie   74 min      1
  Movie   66 min      1
  Movie   84 min      1
TV Show    TV-MA   1145
TV Show    TV-14    733
TV Show    TV-PG    323
TV Show    TV-Y7    195
TV Show     TV-Y    176
TV Show     TV-G     94
TV Show       NR      5
TV Show        R      2
TV Show TV-Y7-FV      1


In [9]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 3. List all movies released in a specific year 2020
    q3 = """
    SELECT title, type, release_year
    FROM NetflixBusinessREV
    WHERE type = 'Movie' AND release_year = 2020
    """
    execute_query(q3, "3. List all movies released in a specific year 2020")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 3. List all movies released in a specific year 2020 ---
                                                                          title  type  release_year
                                                           Dick Johnson Is Dead Movie          2020
                            Europe's Most Dangerous Man: Otto Skorzeny in Spain Movie          2020
                                                                 Tughlaq Durbar Movie          2020
                                                           Omo Ghetto: the Saga Movie          2020
                                                                 Shadow Parties Movie          2020
                                                                 Here and There Movie          2020
                                                                        Shikara Movie          2020
                                                                    

In [10]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 4. Find the top 5 countries with the most content on NetflixBusiness
    q4 = """
    SELECT TOP 5 TRIM(value) as country, COUNT(*) as count
    FROM NetflixBusinessREV
    CROSS APPLY STRING_SPLIT(country, ',')
    WHERE value != '' AND value IS NOT NULL
    GROUP BY TRIM(value)
    ORDER BY count DESC
    """
    execute_query(q4, "4. Top 5 countries with the most content on NetflixBusiness")


if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 4. Top 5 countries with the most content on NetflixBusiness ---
       country  count
 United States   3690
         India   1046
United Kingdom    806
        Canada    445
        France    393


In [11]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 5. Identify the longest movie
    q5 = """
    SELECT TOP 1 title, duration, type
    FROM NetflixBusinessREV
    WHERE type = 'Movie' AND duration IS NOT NULL AND duration != ''
    ORDER BY 
        CAST(LEFT(duration, CHARINDEX(' ', duration+' ')-1) AS INT) DESC
    """
    execute_query(q5, "5. Identify the longest movie")


if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 5. Identify the longest movie ---
                     title duration  type
Black Mirror: Bandersnatch  312 min Movie


In [12]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

     # 6. Find content that was added in the last 5 years
    q6 = """
    SELECT title, date_added, type
    FROM NetflixBusinessREV
    WHERE date_added IS NOT NULL 
    AND YEAR(TRY_CAST(date_added AS DATE)) >= YEAR(GETDATE()) - 5
    """
    execute_query(q6, "6. Find content that was added in the last 5 years")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 6. Find content that was added in the last 5 years ---
                                                                 title         date_added    type
                                                  Dick Johnson Is Dead September 25, 2021   Movie
                                                         Blood & Water September 24, 2021 TV Show
                                                             Ganglands September 24, 2021 TV Show
                                                 Jailbirds New Orleans September 24, 2021 TV Show
                                                          Kota Factory September 24, 2021 TV Show
                                                         Midnight Mass September 24, 2021 TV Show
                                      My Little Pony: A New Generation September 24, 2021   Movie
                                                               Sankofa September 24, 

In [13]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

     # 7. Find all the movies/TV shows by director 'Rajiv Chilaka'
    q7 = """
    SELECT title, type, director
    FROM NetflixBusinessREV
    WHERE director LIKE '%Rajiv Chilaka%'
    """
    execute_query(q7, "7. Find all movies/TV shows by director 'Rajiv Chilaka'")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 7. Find all movies/TV shows by director 'Rajiv Chilaka' ---
                                                title  type                                             director
                         Chhota Bheem - Neeli Pahaadi Movie                                        Rajiv Chilaka
                                Chhota Bheem & Ganesh Movie                                        Rajiv Chilaka
                   Chhota Bheem & Krishna: Mayanagari Movie                                        Rajiv Chilaka
Chhota Bheem & Krishna: Pataliputra- City of the Dead Movie                                        Rajiv Chilaka
                   Chhota Bheem And The Broken Amulet Movie                                        Rajiv Chilaka
               Chhota Bheem And The Crown of Valhalla Movie                                        Rajiv Chilaka
                 Chhota Bheem and the Incan Adventure Movie              

In [14]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 8. List all TV shows with more than 5 seasons
    q8 = """
    SELECT title, duration, type
    FROM NetflixBusinessREV
    WHERE type = 'TV Show' 
    AND duration IS NOT NULL
    AND TRY_CAST(LEFT(duration, CHARINDEX(' ', duration+' ')-1) AS INT) > 5
    """
    execute_query(q8, "8. List all TV shows with more than 5 seasons")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 8. List all TV shows with more than 5 seasons ---
                              title   duration    type
      The Great British Baking Show  9 Seasons TV Show
                          Nailed It  6 Seasons TV Show
                       Numberblocks  6 Seasons TV Show
                  Saved by the Bell  9 Seasons TV Show
                            Lucifer  6 Seasons TV Show
                  Grace and Frankie  7 Seasons TV Show
                            30 Rock  7 Seasons TV Show
             Hunter X Hunter (2011)  6 Seasons TV Show
                          The Flash  7 Seasons TV Show
                   The Walking Dead 10 Seasons TV Show
My Little Pony: Friendship Is Magic  8 Seasons TV Show
            Orange Is the New Black  7 Seasons TV Show
   Terrace House: Opening New Doors  6 Seasons TV Show
                     Grey's Anatomy 17 Seasons TV Show
                               Glee  6 Seasons TV

In [15]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

     # 9. Count the number of content items in each genre (listed_in)
    q9 = """
    SELECT TRIM(value) as genre, COUNT(*) as count
    FROM NetflixBusinessREV
    CROSS APPLY STRING_SPLIT(listed_in, ',')
    WHERE value != '' AND value IS NOT NULL
    GROUP BY TRIM(value)
    ORDER BY count DESC
    """
    execute_query(q9, "9. Count the number of content items in each genre")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 9. Count the number of content items in each genre ---
                       genre  count
        International Movies   2752
                      Dramas   2427
                    Comedies   1674
      International TV Shows   1351
               Documentaries    869
          Action & Adventure    859
                   TV Dramas    763
          Independent Movies    756
    Children & Family Movies    641
             Romantic Movies    616
                 TV Comedies    581
                   Thrillers    577
              Crime TV Shows    470
                    Kids' TV    451
                  Docuseries    395
            Music & Musicals    375
           Romantic TV Shows    370
               Horror Movies    357
             Stand-Up Comedy    343
                  Reality TV    255
            British TV Shows    253
            Sci-Fi & Fantasy    243
               Sports Movies    219
     

In [16]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 10. Find each year and the average numbers of content release by India
    q10 = """
    SELECT TOP 5 release_year, COUNT(*) as content_count
    FROM NetflixBusinessREV
    WHERE country LIKE '%India%'
    AND release_year IS NOT NULL
    GROUP BY release_year
    ORDER BY content_count DESC
    """
    execute_query(q10, "10. Top 5 years with highest content release by India")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 10. Top 5 years with highest content release by India ---
 release_year  content_count
         2017            111
         2018            101
         2019             93
         2016             80
         2020             77


In [17]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 11. List all movies that are documentaries
    q11 = """
    SELECT title, type, listed_in
    FROM NetflixBusinessREV
    WHERE type = 'Movie' AND listed_in LIKE '%Documentary%'
    """
    execute_query(q11, "11. List all movies that are documentaries")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 11. List all movies that are documentaries ---
Empty DataFrame
Columns: [title, type, listed_in]
Index: []


In [18]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 12. Find all content without a director
    q12 = """
    SELECT title, type, director
    FROM NetflixBusinessREV
    WHERE director IS NULL OR director = ''
    """
    execute_query(q12, "12. Find all content without a director")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 12. Find all content without a director ---
                                                                                   title    type director
                                                                           Blood & Water TV Show     None
                                                                   Jailbirds New Orleans TV Show     None
                                                                            Kota Factory TV Show     None
                                                     Vendetta: Truth, Lies and The Mafia TV Show     None
                                                         Crime Stories: India Detectives TV Show     None
                                                                       Dear White People TV Show     None
                                                                         Falsa identidad TV Show     None
                                

In [19]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 13. Find how many movies actor 'Salman Khan' appeared in last 10 years
    q13 = """
    SELECT COUNT(*) as count_movies
    FROM NetflixBusinessREV
    WHERE type = 'Movie' 
    AND cast LIKE '%Salman Khan%'
    AND release_year >= YEAR(GETDATE()) - 10
    """
    execute_query(q13, "13. Find how many movies actor 'Salman Khan' appeared in last 10 years")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 13. Find how many movies actor 'Salman Khan' appeared in last 10 years ---
 count_movies
            1


In [8]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 14. Find the top 10 actors who have appeared in the highest number of movies produced in India
    q14 = """
    SELECT TOP 10 TRIM(value) as actor, COUNT(*) as movie_count
    FROM NetflixBusinessREV
    CROSS APPLY STRING_SPLIT(cast, ',')
    WHERE country LIKE '%India%' AND type = 'Movie'
    AND value IS NOT NULL AND value != ''
    GROUP BY TRIM(value)
    ORDER BY movie_count DESC
    """
    execute_query(q14, "14. Top 10 actors in Indian movies")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 14. Top 10 actors in Indian movies ---
           actor  movie_count
     Anupam Kher           40
  Shah Rukh Khan           34
Naseeruddin Shah           31
         Om Puri           29
    Akshay Kumar           29
Amitabh Bachchan           28
    Paresh Rawal           28
     Boman Irani           27
  Kareena Kapoor           25
      Ajay Devgn           21


In [7]:
def netflix_business_analysis():
    print("NetflixBusiness Data Analysis using Python")
    print("\n-- 15 Business Problems --")

    # 15. Categorize content based on 'kill' and 'violence' in description
    q15 = """
    SELECT 
        CASE 
            WHEN LOWER(ISNULL(description, '')) LIKE '%kill%' OR LOWER(ISNULL(description, '')) LIKE '%violence%' 
            THEN 'Bad' 
            ELSE 'Good' 
        END as category,
        COUNT(*) as count
    FROM NetflixBusinessREV
    GROUP BY 
        CASE 
            WHEN LOWER(ISNULL(description, '')) LIKE '%kill%' OR LOWER(ISNULL(description, '')) LIKE '%violence%' 
            THEN 'Bad' 
            ELSE 'Good' 
        END
    """
    execute_query(q15, "15. Content categorized as 'Bad' or 'Good' based on keywords")

if __name__ == "__main__":
    netflix_business_analysis()

NetflixBusiness Data Analysis using Python

-- 15 Business Problems --

--- 15. Content categorized as 'Bad' or 'Good' based on keywords ---
category  count
    Good   8465
     Bad    342
